In [1]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
!pip install pyarrow==8.0.0  # Downgrade pyarrow to a known compatible version
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset, Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 16.1.0 which is incompatible.
  Using cached pyarrow-8.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (29.4 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 16.1.0
    Uninstalling pyarrow-16.1.0:
      Successfully uninstalled pyarrow-16.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 8.0.0 which is incompatible.
datasets 2.20.0 requires pyarrow>=15.0.0, but you have pyarrow 8.0.0 which is incompatibl

In [2]:

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
tokenizer_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-2
num_epochs = 50
batch_size = 8


In [3]:



# Step 3: Load Dataset

# dataset = pd.read_csv()

# dataset.iloc[0]

column_names = [label_column, "blank", text_column]
# Load the dataset without headers and assign the column names
dataset = pd.read_csv("/content/drive/MyDrive/societal_2l_train_hi.csv", header=None, names=column_names).head(500) # examples in dataset

dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset.head())
# Extract the specific range
# specific_range_dataset = dataset.iloc[1000:1201]  # Adjusted to include index 1200
# print(specific_range_dataset.head())
from sklearn.model_selection import train_test_split

# Split the preprocessed dataset into training and testing sets (80% train, 20% test)
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Now you have train_dataset and test_dataset for training and testing respectively
print(train_dataset)
dataset_train = Dataset.from_pandas(train_dataset)
dataset_test = Dataset.from_pandas(test_dataset)












   text_label                                         Tweet text
0           0  @sardanarohit @sudhirchaudhary @arnabgoswamias...
1           1  @narendramodi प्रिय पीएम, युवाओं को हम #Cashle...
2           0  @Narendramodi सर पठानकोट निगम AMRIT CITY SCHEO...
3           1  #jobs #jobsearch ##parliament GST बिल पास करता...
4           1  @Upsubramanya ppl praise manmohan 4 1991 सुधार...
     text_label                                         Tweet text
249           1  #GST को अच्छी तरह से समझें और इसे अपने निर्वाच...
433           1  @YADAVAKHILESH आप असली SAMAJWADI हैं। ऐसे व्यक...
19            0  यदि पाक के लोग आतंकवाद के खिलाफ खड़े होने के ल...
322           0  Pathankot के दागी SP SALWINDER ने @Timesofindi...
332           0  @Pmoindia, @finminindia @narendramodi केंद्र स...
..          ...                                                ...
106           0  RT @kumar_ke5hav: #surgicalStrike और एक कमजोर ...
270           0  @airtelindia @ideacellular @vodafonein @aircel...
348    

In [4]:
# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs


In [6]:
# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)


Running tokenizer on dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)


In [8]:
# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)  # Binary classification
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 237,565,444 || trainable%: 0.0032
None


In [9]:
# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

In [10]:
# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []  # Initialize a list to collect true labels
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch["labels"].detach().cpu().numpy())  # Collect true labels

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    # Calculate evaluation metrics
    accuracy = accuracy_score(true_labels, eval_preds)
    f1 = f1_score(true_labels, eval_preds, average='weighted')
    recall = recall_score(true_labels, eval_preds, average='weighted')
    precision = precision_score(true_labels, eval_preds, average='weighted')

    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}")


# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})
print(results_df)


100%|██████████| 13/13 [00:00<00:00, 15.81it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 0: train_loss=0.7142744064331055, eval_loss=0.7507162094116211, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25


100%|██████████| 13/13 [00:00<00:00, 15.40it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1: train_loss=0.739112913608551, eval_loss=0.7058295011520386, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25


100%|██████████| 13/13 [00:01<00:00, 10.85it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

100%|██████████| 13/13 [00:00<00:00, 14.51it/s]


Epoch 48: train_loss=0.6946448087692261, eval_loss=0.6931436061859131, accuracy=0.51, f1_score=0.3551783129359126, recall=0.51, precision=0.7525252525252526


100%|██████████| 13/13 [00:00<00:00, 14.59it/s]

Epoch 49: train_loss=0.692729115486145, eval_loss=0.6930314898490906, accuracy=0.5, f1_score=0.33333333333333326, recall=0.5, precision=0.25
    true_label  predicted_label
0            0                1
1            1                1
2            0                1
3            1                1
4            0                1
..         ...              ...
95           0                1
96           0                1
97           1                1
98           1                1
99           1                1

[100 rows x 2 columns]



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [11]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df.head())

Predictions saved to /content/drive/MyDrive/model_predictions.csv
   true_label  predicted_label
0           0                1
1           1                1
2           0                1
3           1                1
4           0                1


In [12]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [13]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.5
F1 Score: 0.33333333333333326
Recall: 0.5
Precision: 0.25


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
